In [1]:
import os
import time
import pandas as pd
from minio import Minio
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types

In [2]:
spark = (
    SparkSession.builder
        .appName("CryptoETL")
        .config("spark.master", "spark://spark-master:7077")
        # ---- Iceberg + Hive Catalog ----
        .config("spark.sql.catalog.hive_catalog", "org.apache.iceberg.spark.SparkCatalog")
        .config("spark.sql.catalog.hive_catalog.catalog-impl", "org.apache.iceberg.hive.HiveCatalog")
        .config("spark.sql.catalog.hive_catalog.uri", "thrift://hive-metastore:9083")
        .config("spark.sql.catalog.hive_catalog.warehouse", "s3a://crypto-data-lake/")
        # ---- Default catalog
        .config("spark.sql.defaultCatalog", "hive_catalog")
        # ---- S3 (MinIO) ----
        .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
        .config("spark.hadoop.fs.s3a.secret.key", "minioadmin")
        .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        # ---- Iceberg Extensions ----
        .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
        .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
        # ---- Extra JARs ----
        .config("spark.jars", ",".join([
            "/opt/spark-extra-jars/iceberg-spark-runtime-3.5_2.12-1.6.1.jar",
            "/opt/spark-extra-jars/hadoop-aws-3.3.4.jar",
            "/opt/spark-extra-jars/aws-java-sdk-bundle-1.12.262.jar"
        ]))
        .getOrCreate()
)

25/09/25 10:34:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [3]:
bucket = "crypto-data-lake"

In [3]:
output_path = f"s3a://{bucket}/landing_zone/spot/daily/aggTrades/BTCUSDT/2025_08_01"
df = spark.read.parquet(output_path)

25/09/23 08:11:09 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
                                                                                

In [4]:
df.show()

+------------+---------+--------+--------------+-------------+----------------+--------------+-------------+-----------+--------------------+
|agg_trade_id|    price|quantity|first_trade_id|last_trade_id|       timestamp|is_buyer_maker|is_best_match|ingest_date|    ingest_timestamp|
+------------+---------+--------+--------------+-------------+----------------+--------------+-------------+-----------+--------------------+
|  3640494121|115764.07| 0.22677|    5122977554|   5122977554|1754006400328945|          true|         true| 2025-09-23|2025-09-23 08:09:...|
|  3640494122|115764.08| 0.00145|    5122977555|   5122977555|1754006400345714|         false|         true| 2025-09-23|2025-09-23 08:09:...|
|  3640494123|115764.08|  2.1E-4|    5122977556|   5122977556|1754006400350235|         false|         true| 2025-09-23|2025-09-23 08:09:...|
|  3640494124|115764.08|  4.1E-4|    5122977557|   5122977557|1754006400492405|         false|         true| 2025-09-23|2025-09-23 08:09:...|
|  364

In [5]:
# .orderBy("timestamp", ascending=True) \
# .sample(0.001) \
# .select(["agg_trade_id", "timestamp", "timestamp_date", "timestamp_second", "group_id", "group_date"]) \
# .show(truncate=False)
df = df.withColumn("timestamp_date", F.from_unixtime(F.col("timestamp") / 1_000_000)) \
    .withColumn("timestamp_second", (F.col("timestamp") / 1_000_000).cast("long")) \
    .withColumn("group_id", (F.col("timestamp_second") / 900).cast("long")) \
    .withColumn("group_date", F.from_unixtime(F.col("group_id") * 900)) \
    .withColumn("transform_date", F.current_date()) \
    .withColumn("transform_timestamp", F.current_timestamp())

In [4]:
spark.sql("""
SHOW DATABASES
""").show()

+------------+
|   namespace|
+------------+
|     default|
|  serving_db|
|transform_db|
+------------+



In [8]:
spark.sql("""
CREATE DATABASE IF NOT EXISTS transform_db
LOCATION 's3a://crypto-data-lake/transform_zone/'
""")

DataFrame[]

In [5]:
spark.sql("""
SHOW TABLES in serving_db
""").show()

+----------+----------+-----------+
| namespace| tableName|isTemporary|
+----------+----------+-----------+
|serving_db|    klines|      false|
|serving_db|test_table|      false|
|serving_db|  klinesv2|      false|
|serving_db|  klinesv4|      false|
|serving_db|  klinesv5|      false|
|serving_db|  klinesv6|      false|
|serving_db|       ma7|      false|
|serving_db|      sma7|      false|
+----------+----------+-----------+



In [12]:
df.writeTo("transform_db.aggtrades").tableProperty("format-version", "2").createOrReplace()

In [14]:
spark.sql("""
select count(*) from transform_db.aggtrades
""").show()

+--------+
|count(1)|
+--------+
| 1314072|
+--------+



In [19]:
spark.sql("""
select 
    group_id,
    group_date,
    max(price) as high_price,
    min(price) as low_price,
    max(agg_trade_id) as max_id,
    min(agg_trade_id) as min_id,
    sum(quantity) as volume
from 
    transform_db.aggtrades
group by group_id, group_date
""").show()

+--------+-------------------+----------+---------+----------+----------+------------------+
|group_id|         group_date|high_price|low_price|    max_id|    min_id|            volume|
+--------+-------------------+----------+---------+----------+----------+------------------+
| 1948919|2025-08-01 05:45:00| 115660.37|115385.92|3640815204|3640806867| 142.2174000000001|
| 1948901|2025-08-01 01:15:00| 115271.13|114638.65|3640653774|3640629264| 448.2353499999814|
| 1948949|2025-08-01 13:15:00| 115916.24|115525.47|3641169452|3641155320|228.84313999998787|
| 1948915|2025-08-01 04:45:00| 115699.58|115485.46|3640781833|3640775020| 132.1634600000024|
| 1948950|2025-08-01 13:30:00| 115679.82|114984.87|3641194791|3641169453| 404.7887299999752|
| 1948912|2025-08-01 04:00:00| 115648.03|115266.64|3640758431|3640746859| 199.9337699999932|
| 1948942|2025-08-01 11:30:00| 115230.76|115056.95|3641075157|3641067981| 137.7531500000022|
| 1948991|2025-08-01 23:45:00|  113400.0|113211.26|3641808192|36418025

In [17]:
spark.sql("""
select 
    agg_trade_id,
    price,
    quantity,
    timestamp
from 
    transform_db.aggtrades
""").show()

+------------+---------+--------+----------------+
|agg_trade_id|    price|quantity|       timestamp|
+------------+---------+--------+----------------+
|  3640494121|115764.07| 0.22677|1754006400328945|
|  3640494122|115764.08| 0.00145|1754006400345714|
|  3640494123|115764.08|  2.1E-4|1754006400350235|
|  3640494124|115764.08|  4.1E-4|1754006400492405|
|  3640494125|115764.08|  8.0E-5|1754006400636161|
|  3640494126|115764.07| 0.00444|1754006400669861|
|  3640494127|115764.07| 0.10877|1754006400781656|
|  3640494128|115764.06|  1.0E-4|1754006400781656|
|  3640494129|115762.89|  1.5E-4|1754006400781656|
|  3640494130|115762.88| 0.28528|1754006400781656|
|  3640494131|115762.67|  5.0E-5|1754006400781656|
|  3640494132|115761.43|   0.023|1754006400781682|
|  3640494133|115761.43|  4.0E-4|1754006400781814|
|  3640494134|115761.43|  0.0231|1754006400782004|
|  3640494135|115761.43|  3.0E-4|1754006400782028|
|  3640494136|115761.43| 0.16322|1754006400782053|
|  3640494137|115761.43| 0.1009

In [38]:
df_kline = spark.sql("""
with groups as (
    select 
        group_id,
        max(price) as high_price,
        min(price) as low_price,
        max(agg_trade_id) as max_id,
        min(agg_trade_id) as min_id,
        sum(quantity) as volume
    from 
        transform_db.aggtrades
    group by group_id
)
select 
    g.group_id,
    a2.group_date,
    a2.timestamp as open_time,
    round(a2.price, 2) as open_price,
    round(g.high_price, 2) as high_price,
    round(g.low_price, 2) as low_price,
    round(a1.price, 2) as close_price,
    round(g.volume, 3) as volume,
    a1.timestamp as close_time
from
    groups g
    JOIN transform_db.aggtrades a1 ON a1.agg_trade_id = g.max_id
    JOIN transform_db.aggtrades a2 ON a2.agg_trade_id = g.min_id
order by g.group_id
""")

In [28]:
spark.sql("""
CREATE DATABASE IF NOT EXISTS serving_db
LOCATION 's3a://crypto-data-lake/serving_zone/'
""")

DataFrame[]

In [39]:
df_kline.show()

+--------+-------------------+----------------+----------+----------+---------+-----------+--------+----------------+
|group_id|         group_date|       open_time|open_price|high_price|low_price|close_price|  volume|      close_time|
+--------+-------------------+----------------+----------+----------+---------+-----------+--------+----------------+
| 1948896|2025-08-01 00:00:00|1754006400328945| 115764.07| 115829.46|115308.55|  115313.01| 302.159|1754007299467573|
| 1948897|2025-08-01 00:15:00|1754007300010950| 115313.01|  115933.0| 115313.0|  115800.01| 450.539|1754008199447993|
| 1948898|2025-08-01 00:30:00|1754008200077603|  115800.0|  115800.0|115423.87|  115517.98| 184.425|1754009099900832|
| 1948899|2025-08-01 00:45:00|1754009100223687| 115517.99| 115527.53|114313.13|  115427.27|1589.765|1754009999974074|
| 1948900|2025-08-01 01:00:00|1754010000363342| 115427.27| 115609.99| 114600.0|   114649.9| 681.883|1754010899995166|
| 1948901|2025-08-01 01:15:00|1754010900041356|  114649.

In [40]:
df_kline.writeTo("serving_db.klines").tableProperty("format-version", "2").createOrReplace()

In [12]:
df.show()

+--------+-------------------+----------------+----------+----------+---------+-----------+-------+----------------+---------+
|group_id|         group_date|       open_time|open_price|high_price|low_price|close_price| volume|      close_time|      ma7|
+--------+-------------------+----------------+----------+----------+---------+-----------+-------+----------------+---------+
| 1948902|2025-08-01 01:30:00|1754011800063081| 115190.37| 115413.91| 115000.0|  115296.45|267.388|1754012699912234|115313.57|
| 1948903|2025-08-01 01:45:00|1754012700223622| 115296.46| 115407.71|115060.11|  115331.86|258.534|1754013599937883|115316.26|
| 1948904|2025-08-01 02:00:00|1754013600005133| 115328.67|  115600.0|115221.07|   115600.0|164.012|1754014499986628|115287.69|
| 1948905|2025-08-01 02:15:00|1754014500063435|  115600.0| 115810.71| 115511.6|  115619.94|168.411|1754015399984518|115302.26|
| 1948906|2025-08-01 02:30:00|1754015400022016| 115619.95|  116019.3|115572.53|  115900.01|221.423|175401629986

In [7]:
df = spark.sql("""
select 
    *
from serving_db.sma7
""")
df.show()

+--------+-------------------+----------------+----------+----------+---------+-----------+-------+----------------+---------+
|group_id|         group_date|       open_time|open_price|high_price|low_price|close_price| volume|      close_time|      ma7|
+--------+-------------------+----------------+----------+----------+---------+-----------+-------+----------------+---------+
| 1948902|2025-08-01 01:30:00|1754011800063081| 115190.37| 115413.91| 115000.0|  115296.45|267.388|1754012699912234|115313.57|
| 1948903|2025-08-01 01:45:00|1754012700223622| 115296.46| 115407.71|115060.11|  115331.86|258.534|1754013599937883|115316.26|
| 1948904|2025-08-01 02:00:00|1754013600005133| 115328.67|  115600.0|115221.07|   115600.0|164.012|1754014499986628|115287.69|
| 1948905|2025-08-01 02:15:00|1754014500063435|  115600.0| 115810.71| 115511.6|  115619.94|168.411|1754015399984518|115302.26|
| 1948906|2025-08-01 02:30:00|1754015400022016| 115619.95|  116019.3|115572.53|  115900.01|221.423|175401629986

In [3]:
df_sorted = (
    spark.sql("SELECT * FROM serving_db.sma7")
    .coalesce(1) # one partition, not shuffle
    .sortWithinPartitions("group_id")
)

25/09/25 10:34:47 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


In [4]:
schema = types.StructType([
    *df_sorted.schema.fields,  # keep all original fields
    types.StructField("ema7", types.DoubleType(), True)
])

In [5]:
from decimal import Decimal, getcontext, ROUND_HALF_UP

# set precision high enough for finance data
getcontext().prec = 28  

def ema_in_chunks(iterator):  # one stream iterator per partition
    alpha = Decimal(2) / Decimal(7 + 1)  # keep alpha as Decimal
    prev = None
    for pdf in iterator:  # 10,000 rows pandas dataframe for a chunk
        ema = []
        for price in pdf["close_price"]:
            price_dec = Decimal(str(price))  # convert to Decimal exactly
            if prev is None:
                prev = Decimal(str(pdf["ma7"].iloc[0]))  # initialize with SMA7
            else:
                prev = alpha * price_dec + (Decimal(1) - alpha) * prev
            # emulate Spark's round(..., 2)
            ema.append(float(prev.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)))
        pdf["ema7"] = ema
        pdf = pdf[[*pdf.columns[:-1], "ema7"]]
        yield pdf

In [7]:
import pyarrow
print(pyarrow.__version__)

15.0.2


In [6]:
ema_df = df_sorted.mapInPandas(ema_in_chunks, schema)

In [7]:
ema_df.show(100)

+--------+-------------------+----------------+----------+----------+---------+-----------+-------+----------------+---------+---------+
|group_id|         group_date|       open_time|open_price|high_price|low_price|close_price| volume|      close_time|      ma7|     ema7|
+--------+-------------------+----------------+----------+----------+---------+-----------+-------+----------------+---------+---------+
| 1948902|2025-08-01 01:30:00|1754011800063081| 115190.37| 115413.91| 115000.0|  115296.45|267.388|1754012699912234|115313.57|115313.57|
| 1948903|2025-08-01 01:45:00|1754012700223622| 115296.46| 115407.71|115060.11|  115331.86|258.534|1754013599937883|115316.26|115318.14|
| 1948904|2025-08-01 02:00:00|1754013600005133| 115328.67|  115600.0|115221.07|   115600.0|164.012|1754014499986628|115287.69|115388.61|
| 1948905|2025-08-01 02:15:00|1754014500063435|  115600.0| 115810.71| 115511.6|  115619.94|168.411|1754015399984518|115302.26|115446.44|
| 1948906|2025-08-01 02:30:00|17540154000

In [8]:
ema_df.writeTo("serving_db.ema7").tableProperty("format-version", "2").createOrReplace()